# Notebook 11 — BISP Coverage Diagnosis

Notebook 04 reports `pct_receives_bisp` missing for 99 of 150 districts in `district_summary.csv`. This notebook diagnoses **why**: aggregation bug (data exists in source files but wasn't combined) vs. a true survey coverage gap (data was never collected for those districts).

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

N_BOOT = 5000
SEED   = 42
rng    = np.random.default_rng(SEED)

os.makedirs('../outputs', exist_ok=True)
os.makedirs('../outputs/figures', exist_ok=True)
print('Setup complete.')

## Step 1 — Check for household.csv

The original diagnosis request assumed a separate `household.csv` might exist with raw BISP responses. Check whether that file is present.

In [ ]:
HOUSEHOLD_PATH = '../Data:/household.csv'

if os.path.exists(HOUSEHOLD_PATH):
    hh = pd.read_csv(HOUSEHOLD_PATH)
    print(f'household.csv found: {hh.shape[0]:,} rows, {hh.shape[1]} columns')
    print('Columns:', list(hh.columns))
    HH_EXISTS = True
else:
    print('household.csv NOT found in Data:/')
    print('Documented data files in Data:/')
    for f in sorted(os.listdir('../Data:/'   if os.path.isdir('../Data:/') else 'Data:/')):
        print(f'  {f}')
    print()
    print('Proceeding with receives_bisp column in child_merged.csv as the source of truth.')
    HH_EXISTS = False

## Step 2 — Analyse `receives_bisp` in `child_merged.csv` by province and district

In [ ]:
df = pd.read_csv('../Data:/child_merged.csv',
                 usecols=['province', 'district', 'receives_bisp'])

print(f'child_merged loaded: {len(df):,} rows')
print(f'receives_bisp: {df["receives_bisp"].notna().sum():,} non-null '
      f'({df["receives_bisp"].isna().sum():,} null = '
      f'{df["receives_bisp"].isna().mean():.1%} missing)\n')

# --- By province ---
prov = df.groupby('province').agg(
    n_rows     = ('receives_bisp', 'size'),
    n_nonnull  = ('receives_bisp', lambda x: x.notna().sum()),
    n_null     = ('receives_bisp', lambda x: x.isna().sum()),
    n_bisp_1   = ('receives_bisp', 'sum'),
    n_bisp_0   = ('receives_bisp', lambda x: (x == 0).sum()),
).assign(frac_null=lambda d: d['n_null'] / d['n_rows'])

print('Province-level receives_bisp coverage:')
print(prov.to_string())
print()

In [ ]:
# --- By district ---
dist = df.groupby(['district', 'province']).agg(
    n_rows    = ('receives_bisp', 'size'),
    n_nonnull = ('receives_bisp', lambda x: x.notna().sum()),
    n_null    = ('receives_bisp', lambda x: x.isna().sum()),
    n_bisp_1  = ('receives_bisp', 'sum'),
).assign(
    frac_null = lambda d: d['n_null'] / d['n_rows'],
    has_bisp_data = lambda d: d['n_nonnull'] > 0
).reset_index()

n_covered   = dist['has_bisp_data'].sum()
n_uncovered = (~dist['has_bisp_data']).sum()
print(f'Districts with receives_bisp data: {n_covered}')
print(f'Districts with zero receives_bisp data: {n_uncovered}')
print()
print('Covered districts by province:')
print(dist[dist['has_bisp_data']].groupby('province')['district'].count().to_string())
print()
print('Uncovered districts by province:')
print(dist[~dist['has_bisp_data']].groupby('province')['district'].count().to_string())

## Step 3 — Cross-reference: missing districts in `district_summary.csv` vs `child_merged.csv`

In [ ]:
ds = pd.read_csv('../Data:/district_summary.csv')

ds_missing_districts = ds[ds['pct_receives_bisp'].isna()]['district'].tolist()
print(f'Districts missing pct_receives_bisp in district_summary: {len(ds_missing_districts)}')

# For each missing district, check child_merged
missing_check = dist[dist['district'].isin(ds_missing_districts)][
    ['district', 'province', 'n_nonnull']
]

has_data_in_child = missing_check[missing_check['n_nonnull'] > 0]
no_data_in_child  = missing_check[missing_check['n_nonnull'] == 0]

print(f'\nOf those 99 missing districts:')
print(f'  Have receives_bisp data in child_merged: {len(has_data_in_child)}')
print(f'  Have ZERO receives_bisp data:            {len(no_data_in_child)}')

if len(has_data_in_child) > 0:
    print('\n>>> AGGREGATION BUG CANDIDATE: data exists but was not aggregated:')
    print(has_data_in_child.to_string(index=False))
else:
    print('\n>>> NO aggregation bug: all 99 missing districts have zero receives_bisp rows in child_merged.')

## Step 4 — Validate recomputed pct_receives_bisp vs district_summary

Recompute the BISP rate from child_merged for the 51 covered districts and compare to `district_summary.csv` to confirm the data pipeline is correct.

In [ ]:
recomp = (
    df.groupby('district')['receives_bisp']
    .agg(n_nonnull=lambda x: x.notna().sum(), bisp_sum='sum')
    .assign(pct_bisp_recomp=lambda d: d['bisp_sum'] / d['n_nonnull'] * 100)
)
recomp = recomp[recomp['n_nonnull'] > 0].reset_index()

val = ds[ds['pct_receives_bisp'].notna()].merge(
    recomp[['district', 'pct_bisp_recomp', 'n_nonnull']],
    on='district', how='left'
)

r_val, p_val = stats.pearsonr(val['pct_receives_bisp'], val['pct_bisp_recomp'])
print(f'Correlation (district_summary vs recomputed): r = {r_val:.4f}, p = {p_val:.2e}')
print(f'Mean absolute difference: {(val["pct_receives_bisp"] - val["pct_bisp_recomp"]).abs().mean():.4f} pp')
print()
print('>>> The district_summary pct_receives_bisp values ARE correctly derived from child_merged.')
print('>>> The 99 missing values reflect true survey non-coverage, not a computation error.')

## Step 5 — Province coverage map

Document which provinces and districts are covered vs. not covered by the BISP question.

In [ ]:
# Coverage summary by province
cov = ds.groupby('province').agg(
    total_districts    = ('district', 'count'),
    covered_districts  = ('pct_receives_bisp', lambda x: x.notna().sum()),
).assign(
    coverage_pct = lambda d: d['covered_districts'] / d['total_districts'] * 100
).sort_values('coverage_pct', ascending=False)

print('BISP survey coverage by province:')
print(cov.to_string())
print()

# BISP rates in covered districts
print('Mean pct_receives_bisp in covered districts by province:')
print(
    ds[ds['pct_receives_bisp'].notna()]
    .groupby('province')['pct_receives_bisp']
    .agg(['mean', 'min', 'max', 'count'])
    .round(1)
    .to_string()
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

cov_plot = cov.reset_index().sort_values('coverage_pct')
colours  = ['#d62728' if v == 0 else '#aec7e8' if v < 50 else '#1f77b4'
            for v in cov_plot['coverage_pct']]

bars = ax.barh(cov_plot['province'], cov_plot['coverage_pct'], color=colours)

for bar, row in zip(bars, cov_plot.itertuples()):
    label = f'{int(row.covered_districts)}/{int(row.total_districts)}'
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            label, va='center', fontsize=9)

ax.set_xlabel('BISP question coverage (% of districts)')
ax.set_title('BISP Survey Coverage by Province\n'
             '(red = 0% coverage; blue = partial or full)')
ax.set_xlim(0, 115)
ax.axvline(50, color='grey', lw=0.8, ls='--')
plt.tight_layout()
plt.savefig('../outputs/figures/11a_bisp_coverage_map.png', dpi=150, bbox_inches='tight')
plt.savefig('../outputs/figures/11a_bisp_coverage_map.pdf', bbox_inches='tight')
plt.show()
print('Saved 11a_bisp_coverage_map.')

## Step 6 — BISP–arithmetic_gap correlation on covered subset (with bootstrap CI)

In [ ]:
bisp_df = ds[ds['pct_receives_bisp'].notna() & ds['arithmetic_gap'].notna()].copy()
n_corr  = len(bisp_df)

r_obs, p_obs = stats.pearsonr(bisp_df['arithmetic_gap'], bisp_df['pct_receives_bisp'])

# Bootstrap CI on r
x_arr = bisp_df['arithmetic_gap'].values
y_arr = bisp_df['pct_receives_bisp'].values

boot_r = []
rng2   = np.random.default_rng(SEED)
for _ in range(N_BOOT):
    idx  = rng2.integers(0, n_corr, n_corr)
    if len(set(idx)) < 3:
        continue
    rv, _ = stats.pearsonr(x_arr[idx], y_arr[idx])
    boot_r.append(rv)

ci_low, ci_high = np.percentile(boot_r, [2.5, 97.5])

print(f'BISP vs arithmetic_gap correlation')
print(f'  n          = {n_corr}')
print(f'  r          = {r_obs:.4f}')
print(f'  p          = {p_obs:.4f}')
print(f'  95% CI     = [{ci_low:.4f}, {ci_high:.4f}]')
print(f'  Provinces  = {bisp_df["province"].value_counts().to_dict()}')
print()

if ci_low > 0:
    print('CI entirely positive: positive association survives bootstrap.')
elif ci_high < 0:
    print('CI entirely negative: negative association survives bootstrap.')
else:
    print('CI crosses zero: association is not robust at 95% confidence.')

In [ ]:
prov_colours = {'KPK': '#2ca02c', 'Sindh': '#1f77b4', 'Punjab': '#ff7f0e',
                'Balochistan': '#d62728'}

fig, ax = plt.subplots(figsize=(7, 5))

for prov, grp in bisp_df.groupby('province'):
    ax.scatter(grp['pct_receives_bisp'], grp['arithmetic_gap'],
               label=prov, color=prov_colours.get(prov, 'grey'),
               alpha=0.7, edgecolors='white', linewidths=0.4, s=50)

# OLS trend line
m, b = np.polyfit(bisp_df['pct_receives_bisp'], bisp_df['arithmetic_gap'], 1)
xr   = np.linspace(bisp_df['pct_receives_bisp'].min(),
                   bisp_df['pct_receives_bisp'].max(), 100)
ax.plot(xr, m * xr + b, color='black', lw=1.5, ls='--', label='OLS fit')

annotation = f'r = {r_obs:.3f}  (p = {p_obs:.3f})\n95% CI [{ci_low:.3f}, {ci_high:.3f}]\nn = {n_corr} districts'
ax.text(0.97, 0.05, annotation, transform=ax.transAxes,
        ha='right', va='bottom', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='grey', alpha=0.8))

ax.set_xlabel('% households receiving BISP')
ax.set_ylabel('Arithmetic gap (private − government)')
ax.set_title('BISP Rate vs Private–Government Arithmetic Gap\n'
             '(covered districts only: KPK, Sindh, Punjab + 1 Balochistan)')
ax.legend(framealpha=0.7, fontsize=9)
ax.axhline(0, color='grey', lw=0.7, ls=':')
plt.tight_layout()
plt.savefig('../outputs/figures/11b_bisp_gap_scatter.png', dpi=150, bbox_inches='tight')
plt.savefig('../outputs/figures/11b_bisp_gap_scatter.pdf', bbox_inches='tight')
plt.show()
print('Saved 11b_bisp_gap_scatter.')

## Verdict

The 99 missing `pct_receives_bisp` values in `district_summary.csv` reflect a **true survey coverage gap**, not an aggregation bug. The BISP survey question (`receives_bisp` in `child_merged.csv`) was simply not collected in those 99 districts: all 99 have exactly zero non-null BISP responses in the underlying child-level data. A separate `household.csv` file does not exist in this project's data directory. The district_summary values for the 51 covered districts are correctly computed (r = 0.984 against a direct recomputation from child_merged).

Coverage is highly uneven: KPK (20/33 districts, 61%), Sindh (15/25, 60%), and Punjab (15/35, 43%) have partial coverage; AJK, Gilgit-Baltistan, and ICT have zero coverage; and Balochistan — the province with the highest dropout rate and steepest private-school advantage — is covered for only 1 of 36 districts (Hub). The BISP–arithmetic_gap correlation (r = 0.292, p = 0.042, n = 49, 95% bootstrap CI [0.065, 0.520]) is statistically significant at the 5% level but the confidence interval nearly spans its full plausible range, reflecting a small and geographically skewed sample. **The r = 0.29 finding survives, but applies only to KPK, Sindh, and Punjab districts and cannot be generalised to Balochistan, AJK, GB, or ICT.** Policy reports should caveat the BISP finding accordingly: it captures the association in areas where BISP enrollment monitoring data exists, while the highest-need province (Balochistan) remains uncharacterised.